In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

# Iniciamos la SparkSession
spark = SparkSession.builder \
    .appName("TestFrank") \
    .getOrCreate()

# Verificamos que podemos leer tu archivo
# Nota: En Docker, tu carpeta local está dentro de 'work'
path = "work/data/clientes.csv"

try:
    df = spark.read.csv(path, header=True, inferSchema=True)
    print("¡Conexión exitosa! Aquí tienes los primeros datos:")
    df.show(5)
except Exception as e:
    print(f"Error al leer el archivo: {e}")
    print("Asegúrate de que el CSV esté en la carpeta data/raw/ dentro de tu proyecto.")

¡Conexión exitosa! Aquí tienes los primeros datos:
+----------+------+-------------+-------+----------+------+------------+----------------+---------------+--------------+------------+----------------+-----------+-----------+---------------+--------------+----------------+--------------------+--------------+------------+-----+
|customerID|gender|SeniorCitizen|Partner|Dependents|tenure|PhoneService|   MultipleLines|InternetService|OnlineSecurity|OnlineBackup|DeviceProtection|TechSupport|StreamingTV|StreamingMovies|      Contract|PaperlessBilling|       PaymentMethod|MonthlyCharges|TotalCharges|Churn|
+----------+------+-------------+-------+----------+------+------------+----------------+---------------+--------------+------------+----------------+-----------+-----------+---------------+--------------+----------------+--------------------+--------------+------------+-----+
|7590-VHVEG|Female|            0|    Yes|        No|     1|          No|No phone service|            DSL|          

In [2]:
# Definimos el mapeo de Inglés a Español
nuevos_nombres = {
    "customerID": "id_cliente",
    "gender": "genero",
    "SeniorCitizen": "adulto_mayor",
    "Partner": "tiene_pareja",
    "Dependents": "tiene_dependientes",
    "tenure": "meses_antiguedad",
    "PhoneService": "servicio_telefonico",
    "MultipleLines": "multiples_lineas",
    "InternetService": "servicio_internet",
    "OnlineSecurity": "seguridad_online",
    "OnlineBackup": "respaldo_online",
    "DeviceProtection": "proteccion_dispositivo",
    "TechSupport": "soporte_tecnico",
    "StreamingTV": "streaming_tv",
    "StreamingMovies": "streaming_peliculas",
    "Contract": "tipo_contrato",
    "PaperlessBilling": "facturacion_electronica",
    "PaymentMethod": "metodo_pago",
    "MonthlyCharges": "cargo_mensual",
    "TotalCharges": "cargo_total",
    "Churn": "fuga_cliente"
}

# Aplicamos el renombramiento masivo
df = df
for col_vieja, col_nueva in nuevos_nombres.items():
    df = df.withColumnRenamed(col_vieja, col_nueva)

# Verificamos el nuevo esquema
print("Esquema estandarizado a español:")
df.printSchema()

Esquema estandarizado a español:
root
 |-- id_cliente: string (nullable = true)
 |-- genero: string (nullable = true)
 |-- adulto_mayor: integer (nullable = true)
 |-- tiene_pareja: string (nullable = true)
 |-- tiene_dependientes: string (nullable = true)
 |-- meses_antiguedad: integer (nullable = true)
 |-- servicio_telefonico: string (nullable = true)
 |-- multiples_lineas: string (nullable = true)
 |-- servicio_internet: string (nullable = true)
 |-- seguridad_online: string (nullable = true)
 |-- respaldo_online: string (nullable = true)
 |-- proteccion_dispositivo: string (nullable = true)
 |-- soporte_tecnico: string (nullable = true)
 |-- streaming_tv: string (nullable = true)
 |-- streaming_peliculas: string (nullable = true)
 |-- tipo_contrato: string (nullable = true)
 |-- facturacion_electronica: string (nullable = true)
 |-- metodo_pago: string (nullable = true)
 |-- cargo_mensual: double (nullable = true)
 |-- cargo_total: string (nullable = true)
 |-- fuga_cliente: strin

In [3]:
df.show(5)

+----------+------+------------+------------+------------------+----------------+-------------------+----------------+-----------------+----------------+---------------+----------------------+---------------+------------+-------------------+--------------+-----------------------+--------------------+-------------+-----------+------------+
|id_cliente|genero|adulto_mayor|tiene_pareja|tiene_dependientes|meses_antiguedad|servicio_telefonico|multiples_lineas|servicio_internet|seguridad_online|respaldo_online|proteccion_dispositivo|soporte_tecnico|streaming_tv|streaming_peliculas| tipo_contrato|facturacion_electronica|         metodo_pago|cargo_mensual|cargo_total|fuga_cliente|
+----------+------+------------+------------+------------------+----------------+-------------------+----------------+-----------------+----------------+---------------+----------------------+---------------+------------+-------------------+--------------+-----------------------+--------------------+-------------+---

In [4]:
# Contar nulos por columna
df.select([
    F.count(F.when(F.col(c).isNull() | (F.col(c) == " "), c))
    .alias(c) for c in df.columns
]).show()

# cargo_total viene como string — convertir a double
df = df.withColumn(
    "cargo_total",
    F.when(F.col("cargo_total") == " ", F.lit(None))
     .otherwise(F.col("cargo_total").cast("double"))
)

# Rellenar nulos en cargo_total con cargo_mensual (clientes nuevos)
df = df.fillna({"cargo_total": df.first()["cargo_mensual"]})

+----------+------+------------+------------+------------------+----------------+-------------------+----------------+-----------------+----------------+---------------+----------------------+---------------+------------+-------------------+-------------+-----------------------+-----------+-------------+-----------+------------+
|id_cliente|genero|adulto_mayor|tiene_pareja|tiene_dependientes|meses_antiguedad|servicio_telefonico|multiples_lineas|servicio_internet|seguridad_online|respaldo_online|proteccion_dispositivo|soporte_tecnico|streaming_tv|streaming_peliculas|tipo_contrato|facturacion_electronica|metodo_pago|cargo_mensual|cargo_total|fuga_cliente|
+----------+------+------------+------------+------------------+----------------+-------------------+----------------+-----------------+----------------+---------------+----------------------+---------------+------------+-------------------+-------------+-----------------------+-----------+-------------+-----------+------------+
|      

In [7]:
# Ver los 11 registros problemáticos
df.filter(F.col("cargo_total").isNull() | (F.col("cargo_total") == " ")).select(
    "id_cliente", "meses_antiguedad", "cargo_mensual", "cargo_total"
).show()

+----------+----------------+-------------+-----------+
|id_cliente|meses_antiguedad|cargo_mensual|cargo_total|
+----------+----------------+-------------+-----------+
+----------+----------------+-------------+-----------+



In [6]:
# Ver los 11 registros problemáticos
df.filter(F.col("cargo_total").isNull() | (F.col("cargo_total") == " ")).select(
    "id_cliente", "meses_antiguedad", "cargo_mensual", "cargo_total"
).show()

# Convertir cargo_total a double (los espacios " " se vuelven null automáticamente)
df = df.withColumn(
    "cargo_total",
    F.col("cargo_total").cast("double")
)

# Rellenar los 11 nulos con cargo_mensual (lógica: cliente nuevo, 1 mes de vida)
df = df.withColumn(
    "cargo_total",
    F.when(F.col("cargo_total").isNull(), F.col("cargo_mensual"))
     .otherwise(F.col("cargo_total"))
)

# Verificar que quedaron en 0
df.select(F.count(F.when(F.col("cargo_total").isNull(), 1)).alias("nulos_restantes")).show()

+----------+----------------+-------------+-----------+
|id_cliente|meses_antiguedad|cargo_mensual|cargo_total|
+----------+----------------+-------------+-----------+
+----------+----------------+-------------+-----------+

+---------------+
|nulos_restantes|
+---------------+
|              0|
+---------------+



In [5]:
# Convertir Yes/No a 1/0 en columnas binarias
cols_binarias = [
    "tiene_pareja", "tiene_dependientes", "servicio_telefonico",
    "seguridad_online", "respaldo_online", "proteccion_dispositivo",
    "soporte_tecnico", "streaming_tv", "streaming_peliculas",
    "facturacion_electronica", "fuga_cliente"
]
for c in cols_binarias:
    df = df.withColumn(c,
        F.when(F.col(c) == "Yes", 1).otherwise(0).cast("integer")
    )

In [8]:
# Resumen estadístico de columnas numéricas
df.select(
    "meses_antiguedad", "cargo_mensual", "cargo_total"
).describe().show()

+-------+------------------+------------------+------------------+
|summary|  meses_antiguedad|     cargo_mensual|       cargo_total|
+-------+------------------+------------------+------------------+
|  count|              7043|              7043|              7043|
|   mean| 32.37114865824223| 64.76169246059922|2279.7809243220254|
| stddev|24.559481023094442|30.090047097678482|2266.7478821865684|
|    min|                 0|             18.25|              18.8|
|    max|                72|            118.75|            8684.8|
+-------+------------------+------------------+------------------+



In [9]:
# Distribución de la variable objetivo
df.groupBy("fuga_cliente") \
  .agg(
    F.count("*").alias("total"),
    F.round(F.count("*") * 100 / df.count(), 1).alias("pct")
  ).show()

+------------+-----+----+
|fuga_cliente|total| pct|
+------------+-----+----+
|           1| 1869|26.5|
|           0| 5174|73.5|
+------------+-----+----+



***Consultas del negocio**

In [10]:
df.groupBy("tipo_contrato") \
  .agg(
    F.count("*").alias("clientes"),
    F.sum("fuga_cliente").alias("fugados"),
    F.round(
        F.mean("fuga_cliente") * 100, 1
    ).alias("tasa_fuga_pct"),
    F.round(F.mean("cargo_mensual"), 2).alias("cargo_prom")
  ) \
  .orderBy("tasa_fuga_pct", ascending=False) \
  .show()

+--------------+--------+-------+-------------+----------+
| tipo_contrato|clientes|fugados|tasa_fuga_pct|cargo_prom|
+--------------+--------+-------+-------------+----------+
|Month-to-month|    3875|   1655|         42.7|      66.4|
|      One year|    1473|    166|         11.3|     65.05|
|      Two year|    1695|     48|          2.8|     60.77|
+--------------+--------+-------+-------------+----------+



In [11]:
#Valor de vida del clinete y perdida  proyectada 

# LTV = cargo_mensual × meses_antiguedad
df = df.withColumn(
    "ltv", F.round(F.col("cargo_mensual") * F.col("meses_antiguedad"), 2)
)

# Revenue mensual en riesgo (clientes que se fueron)
df.groupBy("fuga_cliente") \
  .agg(
    F.round(F.sum("cargo_mensual"), 0).alias("revenue_mensual"),
    F.round(F.avg("ltv"), 0).alias("ltv_promedio"),
    F.round(F.avg("meses_antiguedad"), 1).alias("antiguedad_prom")
  ).show()

+------------+---------------+------------+---------------+
|fuga_cliente|revenue_mensual|ltv_promedio|antiguedad_prom|
+------------+---------------+------------+---------------+
|           1|       139131.0|      1532.0|           18.0|
|           0|       316986.0|      2550.0|           37.6|
+------------+---------------+------------+---------------+



In [14]:
servicios = [
    "seguridad_online", "respaldo_online",
    "proteccion_dispositivo", "soporte_tecnico",
    "streaming_tv", "streaming_peliculas"
]

for svc in servicios:
    print(f"\n--- {svc} ---")
    df.groupBy(svc) \
      .agg(
        F.round(F.mean("fuga_cliente") * 100, 1).alias("tasa_fuga_pct"),
        F.count("*").alias("clientes")
      ) \
      .orderBy(svc) \
      .show()


--- seguridad_online ---
+----------------+-------------+--------+
|seguridad_online|tasa_fuga_pct|clientes|
+----------------+-------------+--------+
|               0|         31.3|    5024|
|               1|         14.6|    2019|
+----------------+-------------+--------+


--- respaldo_online ---
+---------------+-------------+--------+
|respaldo_online|tasa_fuga_pct|clientes|
+---------------+-------------+--------+
|              0|         29.2|    4614|
|              1|         21.5|    2429|
+---------------+-------------+--------+


--- proteccion_dispositivo ---
+----------------------+-------------+--------+
|proteccion_dispositivo|tasa_fuga_pct|clientes|
+----------------------+-------------+--------+
|                     0|         28.7|    4621|
|                     1|         22.5|    2422|
+----------------------+-------------+--------+


--- soporte_tecnico ---
+---------------+-------------+--------+
|soporte_tecnico|tasa_fuga_pct|clientes|
+---------------+----

In [13]:
# Segmentar por antigüedad y cargo
df_seg = df.withColumn("segmento",
    F.when(
        (F.col("meses_antiguedad") >= 24) & (F.col("cargo_mensual") >= 60),
        "VIP — alto valor"
    ).when(
        (F.col("meses_antiguedad") < 6),
        "Nuevo — riesgo alto"
    ).when(
        (F.col("meses_antiguedad") >= 6) & (F.col("cargo_mensual") < 40),
        "Maduro — bajo valor"
    ).otherwise("Medio")
)

df_seg.groupBy("segmento") \
      .agg(
        F.count("*").alias("clientes"),
        F.round(F.mean("fuga_cliente") * 100, 1).alias("tasa_fuga_pct"),
        F.round(F.sum("cargo_mensual"), 0).alias("revenue_mensual")
      ).orderBy("tasa_fuga_pct", ascending=False).show()

+-------------------+--------+-------------+---------------+
|           segmento|clientes|tasa_fuga_pct|revenue_mensual|
+-------------------+--------+-------------+---------------+
|Nuevo — riesgo alto|    1371|         54.3|        74843.0|
|              Medio|    1691|         33.2|       116821.0|
|   VIP — alto valor|    2573|         19.0|       232094.0|
|Maduro — bajo valor|    1408|          5.3|        32358.0|
+-------------------+--------+-------------+---------------+



In [15]:
¿Cuánto se ganaría convirtiendo mes a mes → anual?
# Clientes mes-a-mes activos con perfil estable
candidatos = df.filter(
  (F.col("tipo_contrato") == "Month-to-month") &
  (F.col("meses_antiguedad") > 6) &
  (F.col("fuga_cliente") == 0)
)
candidatos.agg(
  F.count("*").alias("candidatos"),
  F.round(F.sum("cargo_mensual")*12,0).alias("revenue_anual_asegurado"),
  F.round(F.avg("cargo_mensual"),2).alias("cargo_prom")
).show()


Object `anual` not found.
+----------+-----------------------+----------+
|candidatos|revenue_anual_asegurado|cargo_prom|
+----------+-----------------------+----------+
|      1587|              1286087.0|     67.53|
+----------+-----------------------+----------+



In [17]:
#Comportamiento adultos mayores
#¿Se fugan más? ¿Pagan más?

df.groupBy("adulto_mayor") \
  .agg(
    F.count("*").alias("clientes"),
    F.round(F.mean("fuga_cliente")*100,1)
     .alias("tasa_fuga_pct"),
    F.round(F.avg("cargo_mensual"),2)
     .alias("cargo_prom"),
    F.round(F.avg("meses_antiguedad"),1)
     .alias("antiguedad_prom")
  ).show()


+------------+--------+-------------+----------+---------------+
|adulto_mayor|clientes|tasa_fuga_pct|cargo_prom|antiguedad_prom|
+------------+--------+-------------+----------+---------------+
|           1|    1142|         41.7|     79.82|           33.3|
|           0|    5901|         23.6|     61.85|           32.2|
+------------+--------+-------------+----------+---------------+



In [18]:
#Método de pago y fuga
#Electronic check tiene la tasa más alta

df.groupBy("metodo_pago") \
  .agg(
    F.count("*").alias("clientes"),
    F.round(F.mean("fuga_cliente")*100,1)
     .alias("tasa_fuga_pct"),
    F.round(F.avg("cargo_mensual"),2)
     .alias("cargo_prom")
  ) \
  .orderBy("tasa_fuga_pct", ascending=False) \
  .show(truncate=False)

+-------------------------+--------+-------------+----------+
|metodo_pago              |clientes|tasa_fuga_pct|cargo_prom|
+-------------------------+--------+-------------+----------+
|Electronic check         |2365    |45.3         |76.26     |
|Mailed check             |1612    |19.1         |43.92     |
|Bank transfer (automatic)|1544    |16.7         |67.19     |
|Credit card (automatic)  |1522    |15.2         |66.51     |
+-------------------------+--------+-------------+----------+



In [19]:
#¿En qué mes se van más clientes?
# Agrupar en tramos de 12 meses
df.withColumn("cohort",
  F.when(F.col("meses_antiguedad") <= 12, "0-12 meses")
  .when(F.col("meses_antiguedad") <= 24, "13-24 meses")
  .when(F.col("meses_antiguedad") <= 48, "25-48 meses")
  .otherwise("49+ meses")
).groupBy("cohort") \
 .agg(
   F.count("*").alias("clientes"),
   F.round(F.mean("fuga_cliente")*100,1).alias("tasa_fuga_pct")
 ).orderBy("cohort").show()

+-----------+--------+-------------+
|     cohort|clientes|tasa_fuga_pct|
+-----------+--------+-------------+
| 0-12 meses|    2186|         47.4|
|13-24 meses|    1024|         28.7|
|25-48 meses|    1594|         20.4|
|  49+ meses|    2239|          9.5|
+-----------+--------+-------------+

